### Validation with in situ data

#### 1. Packages

In [1]:
# Packages
import contextily as cx
import geopandas as gpd
import glob
import matplotlib.pyplot as plt
import numpy as np
import os
import rioxarray as rxr
from rioxarray.merge import merge_arrays
from scipy.stats import gaussian_kde
from tqdm import tqdm
import xarray as xr
import pandas as pd
from shapely.geometry import box
from scipy import stats

C:\Users\white_rn\AppData\Local\Temp\ipykernel_11152\413517428.py:3: UserWarning: Shapely 2.0 is installed, but because PyGEOS is also installed, GeoPandas will still use PyGEOS by default for now. To force to use and test Shapely 2.0, you have to set the environment variable USE_PYGEOS=0. You can do this before starting the Python process, or in your code before importing geopandas:

import os
os.environ['USE_PYGEOS'] = '0'
import geopandas

In a future release, GeoPandas will switch to using Shapely by default. If you are using PyGEOS directly (calling PyGEOS functions on geometries from GeoPandas), this will then stop working and you are encouraged to migrate from PyGEOS to Shapely 2.0 (https://shapely.readthedocs.io/en/latest/migration_pygeos.html).
  import geopandas as gpd


#### 2. Settings

In [2]:
# File paths to aois
dir_path_aoi_geojsons= r'p:\11209821-cmems-global-sdb\00_miscellaneous\AOIs\*.geojson'
file_path_aoi_geojsons = glob.glob(dir_path_aoi_geojsons)
file_path_aoi_geojsons = [file_path for file_path in file_path_aoi_geojsons if 'adjusted' not in file_path]

# File paths to insitu data
dir_path_insitu_tifs = r'p:\11209821-cmems-global-sdb\00_miscellaneous\Validation_data\*.tif'
file_path_insitu_tifs = glob.glob(dir_path_insitu_tifs)

# File paths to satellite data
dir_path_sat_tifs = r'p:\11209821-cmems-global-sdb\01_intertidal\02_data\05_calibrated\intertidal_improved_100m_benchmark\*.tif'
file_path_sat_tifs = glob.glob(dir_path_sat_tifs)
file_path_sat_tifs = [file_path for file_path in file_path_sat_tifs if os.path.basename(file_path) != 'merged.tif']

# File path with global coastal mask
file_path_global_coastal_mask_parquet = r'p:\11209821-cmems-global-sdb\00_miscellaneous\Feasibility_maps\2023\Buffered\AOI_results\gebco_2023_merge_result.parquet'

# URLs to global lat/hat data (LAT/HAT wrt MSL based on gebco 2023)
file_path_global_lat_zarr = r'https://nx7384.your-storageshare.de/apps/sharingpath/wetwin/public/gebco2023/lat.zarr'
file_path_global_hat_zarr = r'https://nx7384.your-storageshare.de/apps/sharingpath/wetwin/public/gebco2023/hat.zarr'

# Directory to export figures
export_figs = True
dir_path_export_figs = r'p:\11209821-cmems-global-sdb\01_intertidal\04_images\validation'

# Print number of files
print('Number of AOIs:', len(file_path_aoi_geojsons))
print('Number of insitu data files:', len(file_path_insitu_tifs))
print('Number of satellite data files:', len(file_path_sat_tifs))

Number of AOIs: 10
Number of insitu data files: 8
Number of satellite data files: 215


#### 3. Load global data

In [3]:
# Load global coastal mask
print('Load global coastal mask')
gdf_global_coastal_mask = gpd.read_parquet(file_path_global_coastal_mask_parquet)

# Load global lat data
print('Load global lat data')
da_global_lat = xr.open_zarr(file_path_global_lat_zarr)['lat_correction']
da_global_lat = da_global_lat.rename({'lat':'y', 'lon':'x'})
da_global_lat = da_global_lat.rio.write_crs('EPSG:4326')

# Load global hat data
print('Load global hat data')
da_global_hat = xr.open_zarr(file_path_global_hat_zarr)['hat_correction']
da_global_hat = da_global_hat.rename({'lat':'y', 'lon':'x'})
da_global_hat = da_global_hat.rio.write_crs('EPSG:4326')

Load global coastal mask
Load global lat data
Load global hat data


#### 4. Prepare load aois and prepare aoi data

In [4]:
# Get the bounding boxes of the satellite data
da_sat_bboxes = []
for file_path in tqdm(file_path_sat_tifs, desc='Get bboxes of satellite data'):
    # Load the satellite data
    da_sat = rxr.open_rasterio(file_path)

    # Get the bounding box
    bbox = box(*da_sat.rio.bounds())

    # Add the bounding box
    da_sat_bboxes.append(bbox)

Get bboxes of satellite data: 100%|██████████| 215/215 [00:10<00:00, 20.92it/s]


In [5]:
# Load aois
gdf_aoi_ls = []
for file_path in tqdm(file_path_aoi_geojsons, desc='Load aois'):
    # Get the id of the aoi
    id = '_'.join(os.path.basename(file_path).split('.')[0].split('_')[1:])

    # Adjust the id if necessary
    if id == 'NCL_IleDePins':
        id = 'NCL_IledesPins'
    elif id == 'Vir_CruzBay':
        id = 'VIR_CruzBay'

    # Load the insitu geodataframe
    gdf_aoi_ls.append(gpd.read_file(file_path))

    # Add id to the geodataframe
    gdf_aoi_ls[-1]['id'] = id

# Merge insitu geodataframes
gdf_aois = gpd.GeoDataFrame(pd.concat(gdf_aoi_ls, ignore_index=True))
gdf_aois = gdf_aois.set_index('id')

# Initialize dictionary with file paths
insitu_tifs = {}
sat_z9_tifs = {}
sat_z10_tifs = {}
sat_z11_tifs = {}
    
# Add file paths to geodataframe
for id in tqdm(gdf_aois.index, desc='Add file paths to gdf'):
    # Get the insitu file paths
    file_paths = [file_path for file_path in file_path_insitu_tifs if id in file_path]
    
    # Add the insitu file paths
    if len(file_paths) > 0:
        insitu_tifs[id] = file_paths[0]
    else:
        insitu_tifs[id] = None

    # Get the satellite file paths
    file_paths = []
    bboxs = []
    for file_path, bbox in zip(file_path_sat_tifs, da_sat_bboxes):
        if bbox.intersects(gdf_aois.loc[id, 'geometry']):
            file_paths.append(file_path)
            bboxs.append(bbox)
    
    # Add the satellite file paths
    sat_z9_tifs[id] = [file_path for file_path in file_paths if 'z9' in file_path]
    sat_z10_tifs[id] = [file_path for file_path in file_paths if 'z10' in file_path]
    sat_z11_tifs[id] = [file_path for file_path in file_paths if 'z11' in file_path]
    
# Add file paths to geodataframe
gdf_aois = gdf_aois.assign(insitu_tifs=pd.Series(insitu_tifs))
gdf_aois = gdf_aois.assign(sat_z9_tifs=pd.Series(sat_z9_tifs))
gdf_aois = gdf_aois.assign(sat_z10_tifs=pd.Series(sat_z10_tifs))
gdf_aois = gdf_aois.assign(sat_z11_tifs=pd.Series(sat_z11_tifs))

# Print number of files
print('Number of AOIs: {}'.format(len(gdf_aois)))
print('Number of insitu data files: {}'.format(sum(gdf_aois['insitu_tifs'].notna())))
print('Number of satellite data files (z9): {}'.format(sum(gdf_aois['sat_z9_tifs'].map(len))))
print('Number of satellite data files (z10): {}'.format(sum(gdf_aois['sat_z10_tifs'].map(len))))
print('Number of satellite data files (z11): {}'.format(sum(gdf_aois['sat_z11_tifs'].map(len))))
print('Number of satellite data files: {}'.format(sum([sum(gdf_aois['sat_z{}_tifs'.format(z)].map(len)) for z in range(9, 12)])))

# Print geodataframe
gdf_aois

Add file paths to gdf: 100%|██████████| 10/10 [00:06<00:00,  1.64it/s]


Number of AOIs: 10
Number of insitu data files: 8
Number of satellite data files (z9): 31
Number of satellite data files (z10): 45
Number of satellite data files (z11): 113
Number of satellite data files: 189


,geometry,insitu_tifs,sat_z9_tifs,sat_z10_tifs,sat_z11_tifs
id,,,,,
BRA_SaoPaulo,"MULTIPOLYGON (((-46.23325 -24.50150, -46.23300...",p:\11209821-cmems-global-sdb\00_miscellaneous\...,[p:\11209821-cmems-global-sdb\01_intertidal\02...,[p:\11209821-cmems-global-sdb\01_intertidal\02...,[p:\11209821-cmems-global-sdb\01_intertidal\02...
CAN_KingWilliamIsland,"MULTIPOLYGON (((-99.88025 69.41250, -99.88025 ...",p:\11209821-cmems-global-sdb\00_miscellaneous\...,[p:\11209821-cmems-global-sdb\01_intertidal\02...,[p:\11209821-cmems-global-sdb\01_intertidal\02...,[p:\11209821-cmems-global-sdb\01_intertidal\02...
CAN_PrinceRupertIsland,"MULTIPOLYGON (((-130.99175 54.42250, -130.9917...",p:\11209821-cmems-global-sdb\00_miscellaneous\...,[p:\11209821-cmems-global-sdb\01_intertidal\02...,[p:\11209821-cmems-global-sdb\01_intertidal\02...,[p:\11209821-cmems-global-sdb\01_intertidal\02...
GER_WaddenSea,"MULTIPOLYGON (((5.98825 54.46450, 5.98775 54.4...",p:\11209821-cmems-global-sdb\00_miscellaneous\...,[p:\11209821-cmems-global-sdb\01_intertidal\02...,[p:\11209821-cmems-global-sdb\01_intertidal\02...,[p:\11209821-cmems-global-sdb\01_intertidal\02...
IND_Okha,"MULTIPOLYGON (((69.05150 22.35900, 69.05150 22...",None,[p:\11209821-cmems-global-sdb\01_intertidal\02...,[],[p:\11209821-cmems-global-sdb\01_intertidal\02...
KOR_Tean,"MULTIPOLYGON (((125.99450 36.37900, 125.99450 ...",None,[p:\11209821-cmems-global-sdb\01_intertidal\02...,[],[p:\11209821-cmems-global-sdb\01_intertidal\02...
MYT_Mayotte,"MULTIPOLYGON (((44.97083 -12.95833, 44.97083 -...",p:\11209821-cmems-global-sdb\00_miscellaneous\...,[p:\11209821-cmems-global-sdb\01_intertidal\02...,[p:\11209821-cmems-global-sdb\01_intertidal\02...,[p:\11209821-cmems-global-sdb\01_intertidal\02...
NCL_IledesPins,"MULTIPOLYGON (((167.07575 -22.57275, 167.07525...",p:\11209821-cmems-global-sdb\00_miscellaneous\...,[p:\11209821-cmems-global-sdb\01_intertidal\02...,[p:\11209821-cmems-global-sdb\01_intertidal\02...,[p:\11209821-cmems-global-sdb\01_intertidal\02...
VIR_CruzBay,"MULTIPOLYGON (((-64.79775 18.32800, -64.79725 ...",p:\11209821-cmems-global-sdb\00_miscellaneous\...,[p:\11209821-cmems-global-sdb\01_intertidal\02...,[p:\11209821-cmems-global-sdb\01_intertidal\02...,[p:\11209821-cmems-global-sdb\01_intertidal\02...


In [6]:
# Remove AOIs without data
gdf_aois = gdf_aois[gdf_aois['insitu_tifs'].notna() &
                  (gdf_aois['sat_z9_tifs'].map(len) > 0) &
                  (gdf_aois['sat_z10_tifs'].map(len) > 0) &
                  (gdf_aois['sat_z11_tifs'].map(len) > 0)]

# Print number of files
print('Number of AOIs: {}'.format(len(gdf_aois)))
print('Number of insitu data files: {}'.format(sum(gdf_aois['insitu_tifs'].notna())))
print('Number of satellite data files (z9): {}'.format(sum(gdf_aois['sat_z9_tifs'].map(len))))
print('Number of satellite data files (z10): {}'.format(sum(gdf_aois['sat_z10_tifs'].map(len))))
print('Number of satellite data files (z11): {}'.format(sum(gdf_aois['sat_z11_tifs'].map(len))))
print('Number of satellite data files: {}'.format(sum([sum(gdf_aois['sat_z{}_tifs'.format(z)].map(len)) for z in range(9, 12)])))

# Print geodataframe
gdf_aois

Number of AOIs: 8
Number of insitu data files: 8
Number of satellite data files (z9): 28
Number of satellite data files (z10): 45
Number of satellite data files (z11): 100
Number of satellite data files: 173


,geometry,insitu_tifs,sat_z9_tifs,sat_z10_tifs,sat_z11_tifs
id,,,,,
BRA_SaoPaulo,"MULTIPOLYGON (((-46.23325 -24.50150, -46.23300...",p:\11209821-cmems-global-sdb\00_miscellaneous\...,[p:\11209821-cmems-global-sdb\01_intertidal\02...,[p:\11209821-cmems-global-sdb\01_intertidal\02...,[p:\11209821-cmems-global-sdb\01_intertidal\02...
CAN_KingWilliamIsland,"MULTIPOLYGON (((-99.88025 69.41250, -99.88025 ...",p:\11209821-cmems-global-sdb\00_miscellaneous\...,[p:\11209821-cmems-global-sdb\01_intertidal\02...,[p:\11209821-cmems-global-sdb\01_intertidal\02...,[p:\11209821-cmems-global-sdb\01_intertidal\02...
CAN_PrinceRupertIsland,"MULTIPOLYGON (((-130.99175 54.42250, -130.9917...",p:\11209821-cmems-global-sdb\00_miscellaneous\...,[p:\11209821-cmems-global-sdb\01_intertidal\02...,[p:\11209821-cmems-global-sdb\01_intertidal\02...,[p:\11209821-cmems-global-sdb\01_intertidal\02...
GER_WaddenSea,"MULTIPOLYGON (((5.98825 54.46450, 5.98775 54.4...",p:\11209821-cmems-global-sdb\00_miscellaneous\...,[p:\11209821-cmems-global-sdb\01_intertidal\02...,[p:\11209821-cmems-global-sdb\01_intertidal\02...,[p:\11209821-cmems-global-sdb\01_intertidal\02...
MYT_Mayotte,"MULTIPOLYGON (((44.97083 -12.95833, 44.97083 -...",p:\11209821-cmems-global-sdb\00_miscellaneous\...,[p:\11209821-cmems-global-sdb\01_intertidal\02...,[p:\11209821-cmems-global-sdb\01_intertidal\02...,[p:\11209821-cmems-global-sdb\01_intertidal\02...
NCL_IledesPins,"MULTIPOLYGON (((167.07575 -22.57275, 167.07525...",p:\11209821-cmems-global-sdb\00_miscellaneous\...,[p:\11209821-cmems-global-sdb\01_intertidal\02...,[p:\11209821-cmems-global-sdb\01_intertidal\02...,[p:\11209821-cmems-global-sdb\01_intertidal\02...
VIR_CruzBay,"MULTIPOLYGON (((-64.79775 18.32800, -64.79725 ...",p:\11209821-cmems-global-sdb\00_miscellaneous\...,[p:\11209821-cmems-global-sdb\01_intertidal\02...,[p:\11209821-cmems-global-sdb\01_intertidal\02...,[p:\11209821-cmems-global-sdb\01_intertidal\02...
ZAF_WoodyCape,"MULTIPOLYGON (((26.27925 -33.87675, 26.27750 -...",p:\11209821-cmems-global-sdb\00_miscellaneous\...,[p:\11209821-cmems-global-sdb\01_intertidal\02...,[p:\11209821-cmems-global-sdb\01_intertidal\02...,[p:\11209821-cmems-global-sdb\01_intertidal\02...


#### 5. Process aoi data

In [ ]:
# Process data for each aoi
z = 9
for id, row in tqdm(gdf_aois.iterrows(), desc='Process data', total=len(gdf_aois)):
    # === Get geometries ===================================================
    print('Get geometries')
    # Get the aoi
    gdf_aoi = gdf_aois.loc[[id]]

    # Clip the global coastal mask to the aoi
    gdf_coastal_mask = gdf_global_coastal_mask[gdf_global_coastal_mask.intersects(gdf_aoi.geometry[0])]
    gdf_coastal_mask.loc[:, 'geometry'] = gdf_coastal_mask.geometry.map(lambda x: x.intersection(gdf_aoi.geometry[0]))
    gdf_coastal_mask = gdf_coastal_mask.dissolve(by='pixel_value')

    # Get clip polygon for data
    gdf_clip_poly = gpd.GeoDataFrame(geometry=[gdf_coastal_mask.unary_union.buffer(0.05).intersection(gdf_aoi.unary_union)], crs='EPSG:4326')
    clip_poly = gdf_clip_poly.geometry[0]

    # === Load Data =========================================================
    print('Load data')
    # Load the insitu data
    da_insitu = rxr.open_rasterio(row['insitu_tifs']).isel(band=0)
    da_insitu = da_insitu.where(da_insitu != -9999)
    da_insitu = da_insitu.rio.set_nodata(np.nan)

    # Load the satellite data
    da_sat_ls = []
    for file_path in row['sat_z{}_tifs'.format(z)]:
        da_sat = rxr.open_rasterio(file_path).isel(band=0)
        da_sat_ls.append(da_sat)
    
    # Merge the satellite data
    da_sat = merge_arrays(da_sat_ls, nodata=np.nan)

    # === Clip Data =========================================================
    print('Clip data')
    # Clip the lat data to the aoi
    da_lat = da_global_lat.rio.clip_box(*clip_poly.bounds)

    # Clip the hat data to the aoi
    da_hat = da_global_hat.rio.clip_box(*clip_poly.bounds)
    
    # Clip the insitu data to the aoi
    da_insitu = da_insitu.rio.clip_box(*clip_poly.bounds)
    da_insitu = da_insitu.rio.clip([clip_poly])

    # Clip the satellite data to the aoi
    da_sat = da_sat.rio.clip_box(*clip_poly.bounds)
    da_sat = da_sat.rio.clip([clip_poly])
    
    # === Match Data ========================================================
    print('Match data')
    # Match the lat data to the satellite data
    da_lat = da_lat.rio.reproject_match(da_sat)

    # Match the insitu data to the satellite data
    da_insitu = da_insitu.rio.reproject_match(da_sat)

    # === Compare Data ======================================================
    print('Compare data')
    # Convert insitu data from LAT to MSL
    da_insitu = da_insitu - da_lat

    # Clip insitu data based on LAT and HAT range
    lat_min = da_lat.min().values
    hat_max = da_hat.max().values
    da_insitu = da_insitu.where((da_insitu > lat_min) & (da_insitu < hat_max))

    # Calculate the difference between the satellite and insitu data
    da_diff = da_sat - da_insitu
    
    # === Calculate Statistics ==============================================
    print('Calculate statistics')
    # Get insitu and satellite data
    zs_insitu = da_insitu.values.flatten()
    zs_sat = da_sat.values.flatten()

    # Remove nans
    idx_nans = np.logical_or(np.isnan(zs_insitu), np.isnan(zs_sat))
    zs_insitu = zs_insitu[~idx_nans]
    zs_sat = zs_sat[~idx_nans]

    # Get statistics
    if len(zs_insitu) > 2:
        spearman_r = stats.spearmanr(zs_insitu, zs_sat).correlation # Spearman rank-order correlation coefficient (1 = perfect correlation)
        pearson_r = stats.pearsonr(zs_insitu, zs_sat)[0]            # Pearson correlation coefficient (1 = perfect correlation)
        r2 = stats.linregress(zs_insitu, zs_sat).rvalue**2          # Coefficient of determination (1 = perfect correlation)
        rmse = np.sqrt(np.mean((zs_insitu - zs_sat)**2))            # Root Mean Squared Error (lower is better)
        mae = np.mean(np.abs(zs_insitu - zs_sat))                   # Mean Absolute Error (lower is better)
    else:
        spearman_r = np.nan
        pearson_r = np.nan
        r2 = np.nan
        rmse = np.nan
        mae = np.nan

    # Get color for scatter
    if len(zs_sat) > 2:
        xy = np.vstack([zs_insitu, zs_sat])
        zs_color = gaussian_kde(xy)(xy)
    else:
        zs_color = np.array(['white']*len(zs_sat))
    
    # Split dataset in two parts
    zs_insitu_red = zs_insitu[np.where(zs_sat > zs_insitu)]
    zs_sat_red = zs_sat[np.where(zs_sat > zs_insitu)]
    zs_color_red = zs_color[np.where(zs_sat > zs_insitu)]
    zs_insitu_blue = zs_insitu[np.where(zs_sat <= zs_insitu)]
    zs_sat_blue = zs_sat[np.where(zs_sat <= zs_insitu)]
    zs_color_blue = zs_color[np.where(zs_sat <= zs_insitu)]

    # === Plot Data =========================================================
    print('Plot data')
    # Get bounds
    bounds = clip_poly.buffer(0.01).bounds

    # Plot map
    fig, ax = plt.subplots(1, 1, figsize=(16, 16*(bounds[3]-bounds[1])/(bounds[2]-bounds[0])))
    gdf_aoi.plot(ax=ax, edgecolor='white', facecolor='none')
    gdf_coastal_mask.plot(ax=ax, edgecolor='blue', facecolor='none')
    gdf_clip_poly.plot(ax=ax, edgecolor='red', facecolor='none')
    da_diff.plot.imshow(ax=ax, vmin=-1.5, vmax=1.5, cmap='RdBu_r', cbar_kwargs={'label': 'Difference between satellite and insitu elevation [m]'})
    
    # Set extent
    ax.set_xlim([bounds[0], bounds[2]])
    ax.set_ylim([bounds[1], bounds[3]])
    ax.set_aspect('equal')
    
    # Add basemap
    cx.add_basemap(ax, crs='EPSG:4326', source=cx.providers.Esri.WorldImagery, zorder=-1)

    # Legend for mask and aoi
    ax.add_patch(plt.Polygon(np.array([[np.nan, np.nan]]), label='AOI', edgecolor='white', facecolor='none'))
    ax.add_patch(plt.Polygon(np.array([[np.nan, np.nan]]), label='Coastal mask', edgecolor='blue', facecolor='none'))
    ax.add_patch(plt.Polygon(np.array([[np.nan, np.nan]]), label='Coastal mask (buffered)', edgecolor='red', facecolor='none'))
    ax.legend()

    # Format figure
    ax.set_xlabel('Longitude WGS 84 [deg]')
    ax.set_ylabel('Latitude WGS 84 [deg]')
    ax.set_title('{} ({})'.format(' '.join(id.split('_')[1:]).title(), id.split('_')[0].upper()))
    fig.tight_layout()

    # Export figure
    if export_figs:
        fig.savefig(os.path.join(dir_path_export_figs, '{}_map_z{}.png'.format(id, z)), dpi=300, bbox_inches='tight')
        plt.close(fig)
    else:
        fig.show()

    # Plot statistics
    fig, ax = plt.subplots(1, 1, figsize=(8, 8))
    ax.scatter(zs_insitu_red, zs_sat_red, c=zs_color_red, s=10, edgecolor='none', cmap='Reds')
    ax.scatter(zs_insitu_blue, zs_sat_blue, c=zs_color_blue, s=10, edgecolor='none', cmap='Blues')
    ax.scatter(np.nan, np.nan, c='red', s=10, edgecolor='none', label='Satellite > Insitu')
    ax.scatter(np.nan, np.nan, c='blue', s=10, edgecolor='none', label='Satellite <= Insitu')
    ax.plot([lat_min-0.1, hat_max+0.1], [lat_min-0.1, hat_max+0.1], color='black', linestyle='--')
    ax.vlines(lat_min, lat_min, hat_max, color='blue', linestyle='--', label='Minimum LAT')
    ax.vlines(hat_max, lat_min, hat_max, color='red', linestyle='--', label='Maximum HAT')
    ax.hlines(lat_min, lat_min, hat_max, color='blue', linestyle='--')
    ax.hlines(hat_max, lat_min, hat_max, color='red', linestyle='--')

    # Add text
    text = '\n'.join(['Spearman $\\rho$ {:.2f}'.format(spearman_r),
                      'Pearson $\\rho$: {:.2f}'.format(pearson_r),
                      'R²:        {:.2f}'.format(r2),
                      'RMSE:      {:.2f} m'.format(rmse),
                      'MAE:       {:.2f} m'.format(mae)])
    ax.text(0.02, 0.978, text, color='black', transform=ax.transAxes, verticalalignment='top', family='monospace', linespacing=1.5,
            bbox=dict(facecolor='white', edgecolor='grey', boxstyle='round,pad=0.2'))
    
    # Format figure
    ax.set_xlim([lat_min-0.1, hat_max+0.1])
    ax.set_ylim([lat_min-0.1, hat_max+0.1])
    ax.set_xlabel('Insitu elevation [m+MSL]')
    ax.set_ylabel('Satellite elevation [m+MSL]')
    ax.set_title('{} ({})'.format(' '.join(id.split('_')[1:]).title(), id.split('_')[0].upper()))
    ax.set_aspect('equal')
    ax.legend(loc='lower right', framealpha=1)
    ax.grid()
    fig.tight_layout()

    # Export figure
    if export_figs:
        fig.savefig(os.path.join(dir_path_export_figs, '{}_scatter_z{}.png'.format(id, z)), dpi=300, bbox_inches='tight')
        plt.close(fig)
    else:
        fig.show()

    # Plot histograms
    fig, axs = plt.subplots(1, 2, figsize=(16, 8))
    axs[0].hist(zs_insitu, bins=100, color='grey', label='Insitu elevation')
    axs[1].hist(zs_sat, bins=100, color='grey', label='Satellite elevation')
    
    # Format figure
    for ax in axs:
        ylims = ax.get_ylim()
        ax.vlines(lat_min, 0, ylims[1], color='blue', linestyle='--', label='Minimum LAT')
        ax.vlines(hat_max, 0, ylims[1], color='red', linestyle='--', label='Maximum HAT')
        ax.set_xlabel('Elevation [m+MSL]')
        ax.set_ylabel('Frequency [-]')
        ax.set_title('{} ({})'.format(' '.join(id.split('_')[1:]).title(), id.split('_')[0].upper()))
        ax.set_xlim(lat_min-0.1, hat_max+0.1)
        ax.set_ylim(0, ylims[1])
        ax.legend()
        ax.grid()
    fig.tight_layout()

    # Export figure
    if export_figs:
        fig.savefig(os.path.join(dir_path_export_figs, '{}_histogram_z{}.png'.format(id, z)), dpi=300, bbox_inches='tight')
        plt.close(fig)
    else:
        fig.show()

    # Clear memory
    del da_insitu, da_sat, da_diff, da_lat, gdf_coastal_mask, gdf_clip_poly, gdf_aoi

Process data:   0%|          | 0/8 [00:00<?, ?it/s]

Get geometries
Load data
Clip data
Match data
Compare data
Calculate statistics
Plot data


Process data:  12%|█▎        | 1/8 [00:20<02:24, 20.66s/it]

Get geometries
Load data
Clip data
Match data
Compare data
Calculate statistics
Plot data


Process data:  25%|██▌       | 2/8 [01:36<05:17, 52.95s/it]

...

Process data:  75%|███████▌  | 6/8 [07:08<02:02, 61.17s/it]

Get geometries
Load data
Clip data
Match data
Compare data
Calculate statistics
Plot data


Process data:  88%|████████▊ | 7/8 [08:20<01:04, 64.74s/it]

Get geometries
Load data
Clip data
Match data
Compare data
Calculate statistics
Plot data


Process data: 100%|██████████| 8/8 [08:59<00:00, 67.41s/it]
